# 00 · Colab setup
Run this once per new Colab environment (Runtime → Change runtime type → **A100 GPU**).
It mounts Drive, installs the repo and checks that everything imports.
**Paste back:** the version block printed at the end, so it can be recorded in `docs/DECISIONS.md`.

## Setup
Mounts Drive, clones or updates the repo, installs it and sets the `SOPVERIFY_*` paths. Safe to re-run after a runtime reset.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
%%bash
set -e
REPO=/content/SOP-Verification
BRANCH=claude/nifty-mccarthy-5suut3   # switch to main once the branch is merged
if [ -d $REPO/.git ]; then
  git -C $REPO fetch -q origin $BRANCH && git -C $REPO checkout -q $BRANCH && git -C $REPO pull -q --ff-only origin $BRANCH
else
  git clone -q -b $BRANCH https://github.com/RohanSada/SOP-Verification.git $REPO
fi
pip install -q -e "$REPO[app,vlm,dev]"
git -C $REPO log --oneline -1

In [ ]:
import os, sys
sys.path.insert(0, "/content/SOP-Verification/src")   # editable install without a runtime restart
os.chdir("/content/SOP-Verification")
from sopverify.colab import setup_env, show
ROOT = "/content/drive/MyDrive/sopverify"
ENV = setup_env(ROOT)
try:  # optional Hugging Face token (Colab secret named HF_TOKEN), in case the dataset is gated
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass
ENV

In [ ]:
!sopverify --help
!sopverify info

In [ ]:
!python -m pytest -q -m "not data and not gpu" 2>&1 | tail -3

In [ ]:
import importlib, platform, subprocess
pkgs = ["torch", "transformers", "numpy", "scipy", "pandas", "sklearn", "pydantic", "av", "cv2", "plotly", "gradio",
        "networkx", "huggingface_hub"]
print("python", platform.python_version())
for p in pkgs:
    try:
        m = importlib.import_module(p); print(p, getattr(m, "__version__", "?"))
    except Exception as e:
        print(p, "NOT INSTALLED", e)
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)